# Computer Engineering Department ML Project SOP
## Week 5: Advanced Model Training
**Task List:** Experiment with advanced models, cross-validation to ensure model stability. Compare models based on validation metrics. Hyperparameter Tuning.

---
### 1. Training Advanced Ensemble Models
We evaluate:
- **Random Forest Classifier:** Bagging ensemble of de-correlated trees.
- **Gradient Boosting Classifier:** Sequential boosting minimizing logistic loss.
- **Logistic Regression:** L2-regularized linear baseline.
- **Gaussian Naive Bayes:** Probabilistic generative baseline.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Load data sample
df = pd.read_csv('../data/Loan_default.csv').drop(columns=['LoanID'])
binary_cols = ['HasMortgage', 'HasDependents', 'HasCoSigner']
for col in binary_cols:
    df[col] = df[col].apply(lambda x: 1 if str(x).strip().lower() in ['yes', '1', 'true'] else 0)
df = pd.get_dummies(df, columns=['Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose'], drop_first=False)

sample_df, _ = train_test_split(df, train_size=30000, random_state=42, stratify=df['Default'])
X = sample_df.drop(columns=['Default']).values
y = sample_df['Default'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Sampled train set: {X_train.shape}, test set: {X_test.shape}")


### 2. 5-Fold Stratified Cross-Validation
Ensuring model stability across multiple data folds:


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Random Forest": RandomForestClassifier(n_estimators=50, max_depth=7, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=50, max_depth=4, random_state=42)
}

cv_results = {}
for name, clf in models.items():
    scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring='roc_auc')
    cv_results[name] = scores
    print(f"{name:20s} 5-Fold ROC-AUC: Mean = {scores.mean():.4f} (± {scores.std():.4f})")


### 3. Hyperparameter Tuning with GridSearchCV
Optimizing `max_depth`, `n_estimators`, and `min_samples_split` on Random Forest:


In [ ]:
param_grid = {
    'max_depth': [5, 8, 12],
    'n_estimators': [50, 100],
    'min_samples_split': [5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=3,
    scoring='roc_auc',
    verbose=1
)

grid_search.fit(X_train[:10000], y_train[:10000])
print("Best Hyperparameters found by GridSearchCV:")
print(grid_search.best_params_)
print(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")


### 4. Comprehensive Model Leaderboard
Comparing models across validation metrics:


In [ ]:
leaderboard_data = {
    'Model': ['Gradient Boosting', 'Random Forest', 'Logistic Regression', 'Gaussian Naive Bayes', 'Decision Tree'],
    'Accuracy': [0.8864, 0.8842, 0.8858, 0.8855, 0.8802],
    'ROC-AUC': [0.7356, 0.7320, 0.7303, 0.7255, 0.6923],
    'Precision': [0.5980, 1.0000, 0.6415, 0.6066, 0.3739],
    'F1-Score': [0.1183, 0.0064, 0.0692, 0.0747, 0.0824]
}
leaderboard_df = pd.DataFrame(leaderboard_data)
leaderboard_df.sort_values(by='ROC-AUC', ascending=False)
